# Does scale or confound removal fix the probe battery? (Q1.H2.E6 and E7)

After every methodological repair, no emotion-probe configuration passed the registered battery
test. Two explanations remained. This notebook scores both.

1. **Scale (E6)**: probe directions were noise-limited at 9 to 16 stories per emotion. We now
   have 256 stories per emotion, generated by the probed model itself (leakage 1.4%).
2. **Confound removal (E7)**: the paper projects out the top principal components (PCs, from
   principal component analysis, PCA) of activations on emotionally neutral transcripts before
   using emotion vectors as probes. This step was missing from the reference code and from our
   pipeline until yesterday.

Registered predictions (TREE.md, committed before scoring): E6 predicts battery scores rise
with n and some configuration passes 8 of 12 on both batteries by n=256. E7 predicts the
projection restores valence toward PC1 on the instruct model and improves battery scores at
matched n. Pass bar for any configuration: at least 8 of 12 scenarios rank their target
emotion in the top 3 probes, on the paper battery AND our held-out battery.

Data: `results/e6_scale_means.npz` (subsampled probe means at n in 16/64/128/256),
`results/e7_neutral_bundle.npz` (128 neutral-transcript vectors),
`results/probe_sweep_it/` (battery activations, chat and plain formats).

In [1]:
# this cell loads all inputs and defines the one scoring function used throughout
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import rankdata

from emotion_vectors.analysis import project_out_neutral

ROOT = Path("..")
e6 = np.load(ROOT / "results/e6_scale_means.npz", allow_pickle=True)
MEANS = e6["means"].astype(np.float32)  # [n_bucket, emotion, layer, d_model]
N_BUCKETS = list(e6["n_buckets"])
EMOTIONS = list(map(str, e6["emotions"]))
LAYERS = list(e6["layers"])
neutral = np.load(ROOT / "results/e7_neutral_bundle.npz")["vectors"].astype(np.float32)

sweep = np.load(ROOT / "results/probe_sweep_it/activations.npz", allow_pickle=True)
FORMATS = list(map(str, sweep["formats"]))
prompts = [json.loads(l) for l in open(ROOT / "results/probe_sweep_it/prompts.jsonl")]
BATTERIES = {k: [i for i, p in enumerate(prompts) if p["kind"] == k] for k in ("scenario", "heldout")}
READOUTS = ["last", "mean_all", "mean_content"]

def top3(probe_means_ed, fmt, kind, readout, lp):
    """Scenarios (of 12) whose target emotion ranks top-3 among the 12 probes."""
    acts = sweep[f"{fmt}_{readout}"].astype(np.float32)
    P = probe_means_ed - probe_means_ed.mean(axis=0)
    P /= np.linalg.norm(P, axis=1, keepdims=True)
    A = np.stack([acts[i, lp] for i in BATTERIES[kind]])
    M = P @ A.T / np.linalg.norm(A, axis=1)
    return sum(int(12 - rankdata(M[:, j])[j]) + 1 <= 3 for j in range(12)), M

print(f"{MEANS.shape=} | formats {FORMATS} | {len(LAYERS)} layers")

MEANS.shape=(4, 12, 20, 5376) | formats ['plain', 'chat'] | 20 layers


## 1. E6: the scale curve

How to read: each line is one battery (solid = paper's scenarios, dashed = our held-out set).
The x axis is stories per emotion used to build the probes; the y axis is the best battery
score across all format x layer x readout combinations at that n (best-of is fair here because
the same selection is applied at every n; the pass bar of 8 is the horizontal line). A rising
curve crossing the bar vindicates the scale hypothesis; a plateau exonerates scale.

In [2]:
# this cell scores the battery at every n and plots the scale curve
def best_scores(probe_means_layers):  # [emotion, layer, d_model] -> per-battery best + argmax
    best = {"scenario": 0, "heldout": 0}
    best_cfg, joint_best = None, -1
    for fmt in FORMATS:
        for lp in range(len(LAYERS)):
            for r in READOUTS:
                s, _ = top3(probe_means_layers[:, lp, :], fmt, "scenario", r, lp)
                h, _ = top3(probe_means_layers[:, lp, :], fmt, "heldout", r, lp)
                best["scenario"] = max(best["scenario"], s)
                best["heldout"] = max(best["heldout"], h)
                if min(s, h) > joint_best:
                    joint_best, best_cfg = min(s, h), (fmt, LAYERS[lp], r, s, h)
    return best, best_cfg

scale_rows = []
for ni, n in enumerate(N_BUCKETS):
    best, cfg = best_scores(MEANS[ni])
    scale_rows.append((n, best["scenario"], best["heldout"], cfg))
    print(f"n={n:3d}: best paper {best['scenario']}/12, best heldout {best['heldout']}/12, "
          f"best joint config {cfg}")

fig = go.Figure()
fig.add_scatter(x=[r[0] for r in scale_rows], y=[r[1] for r in scale_rows],
                mode="lines+markers", name="paper battery (best)")
fig.add_scatter(x=[r[0] for r in scale_rows], y=[r[2] for r in scale_rows],
                mode="lines+markers", name="held-out battery (best)", line=dict(dash="dash"))
fig.add_hline(y=8, line_dash="dot", annotation_text="registered pass bar (8/12)")
fig.update_layout(title="E6: battery score vs probe-corpus size",
                  xaxis_title="stories per emotion (log scale)", xaxis_type="log",
                  yaxis_title="scenarios with target in top-3 (of 12)",
                  yaxis_range=[0, 12], height=420)
fig.show()

n= 16: best paper 5/12, best heldout 6/12, best joint config ('chat', np.int64(39), 'last', 5, 5)


n= 64: best paper 5/12, best heldout 4/12, best joint config ('plain', np.int64(42), 'mean_content', 4, 4)


n=128: best paper 5/12, best heldout 5/12, best joint config ('chat', np.int64(57), 'last', 5, 5)


n=256: best paper 4/12, best heldout 5/12, best joint config ('plain', np.int64(6), 'mean_all', 4, 4)


## 2. E6: do probe directions converge with n?

How to read: for each n, the cosine between that n's contrast direction and the n=256
direction, averaged over the 12 emotions. Rising toward 1.0 means more data buys more stable
probes (prediction P1). This is the convergence version of the bootstrap-stability check
(claim C2 measured self-cosine 0.877 at n of about 9).

In [3]:
# this cell measures probe-direction convergence toward the n=256 reference
LP33 = LAYERS.index(33)
ref = MEANS[-1, :, LP33, :] - MEANS[-1, :, LP33, :].mean(axis=0)
conv = []
for ni, n in enumerate(N_BUCKETS[:-1]):
    cur = MEANS[ni, :, LP33, :] - MEANS[ni, :, LP33, :].mean(axis=0)
    cos = (cur * ref).sum(1) / (np.linalg.norm(cur, axis=1) * np.linalg.norm(ref, axis=1))
    conv.append(float(cos.mean()))
    print(f"n={n:3d} vs n=256: mean contrast-direction cosine {cos.mean():.3f} (min {cos.min():.3f})")
fig = go.Figure(go.Scatter(x=N_BUCKETS[:-1], y=conv, mode="lines+markers"))
fig.update_layout(title="E6: probe-direction convergence with corpus size (layer 33)",
                  xaxis_title="stories per emotion (log scale)", xaxis_type="log",
                  yaxis_title="cosine vs n=256 direction", height=380)
fig.show()

n= 16 vs n=256: mean contrast-direction cosine 0.944 (min 0.894)
n= 64 vs n=256: mean contrast-direction cosine 0.991 (min 0.983)
n=128 vs n=256: mean contrast-direction cosine 0.997 (min 0.993)


## 3. E7: neutral-PC projection

The paper's confound-removal step: PCA on 128 neutral-transcript activations, components up to
50% cumulative variance projected out of the probes.

How to read the table: battery scores with unprojected vs projected probes, at n=256, for the
three registered layers plus the best sweep cell. Improvement in the projected column supports
prediction P2. The geometry check below it tests P1: does valence return to PC1 after
projection?

In [4]:
# this cell scores projected vs unprojected probes and checks the valence-PC position
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad

n256 = MEANS[-1]  # [emotion, layer, d_model]
proj_means = np.zeros_like(n256)
removed = {}
for lp in range(len(LAYERS)):
    proj_means[:, lp, :], k = project_out_neutral(n256[:, lp, :], neutral[:, lp, :])
    removed[LAYERS[lp]] = k
print("neutral PCs removed per layer (50% variance):", removed)

print(f"\n{'probes':12s} {'fmt':6s} {'layer':5s} {'readout':13s} {'paper':>6s} {'heldout':>8s}")
for name, pm in (("unprojected", n256), ("projected", proj_means)):
    for fmt in FORMATS:
        for layer in (33, 39, 57):
            lp = LAYERS.index(layer)
            for r in ("last", "mean_content"):
                s, _ = top3(pm[:, lp, :], fmt, "scenario", r, lp)
                h, _ = top3(pm[:, lp, :], fmt, "heldout", r, lp)
                if s + h >= 8 or (layer == 57 and r == "mean_content"):
                    print(f"{name:12s} {fmt:6s} {layer:5d} {r:13s} {s:4d}/12 {h:6d}/12")

best_unproj, cfg_u = best_scores(n256)
best_proj, cfg_p = best_scores(proj_means)
print(f"\nbest joint (unprojected): {cfg_u}")
print(f"best joint (projected):   {cfg_p}")
PASS = [c for c in [cfg_u, cfg_p] if c and c[3] >= 8 and c[4] >= 8]
print(f"registered rule, either probe set: {'PASS ' + str(PASS) if PASS else 'NONE pass'}")

neutral PCs removed per layer (50% variance): {np.int64(0): 4, np.int64(3): 6, np.int64(6): 7, np.int64(9): 7, np.int64(12): 3, np.int64(15): 4, np.int64(18): 4, np.int64(21): 4, np.int64(24): 5, np.int64(27): 8, np.int64(30): 7, np.int64(33): 5, np.int64(36): 5, np.int64(39): 4, np.int64(42): 4, np.int64(45): 3, np.int64(48): 3, np.int64(51): 3, np.int64(54): 4, np.int64(57): 4}

probes       fmt    layer readout        paper  heldout


unprojected  plain     57 mean_content     3/12      3/12


unprojected  chat      57 last             4/12      5/12
unprojected  chat      57 mean_content     4/12      3/12


projected    plain     57 mean_content     4/12      4/12
projected    chat      39 last             5/12      5/12


projected    chat      39 mean_content     4/12      4/12
projected    chat      57 last             4/12      6/12
projected    chat      57 mean_content     5/12      4/12



best joint (unprojected): ('plain', np.int64(6), 'mean_all', 4, 4)
best joint (projected):   ('chat', np.int64(18), 'mean_all', 5, 5)
registered rule, either probe set: NONE pass


In [5]:
# this cell checks whether projection moves valence back toward PC1 (prediction P1)
vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
it_all = np.load(ROOT / "results/emotion_vectors_it_means.npz", allow_pickle=True)
all_emotions, all_means = list(map(str, it_all["emotions"])), it_all["means"].astype(np.float64)
matched = [i for i, e in enumerate(all_emotions) if e.lower() in vad]
valence = np.array([vad[all_emotions[i].lower()][0] for i in matched])
LP = LAYERS.index(33)

for name, M171 in (("unprojected", all_means[:, LP, :]),
                   ("projected", project_out_neutral(all_means[:, LP, :], neutral[:, LP, :].astype(np.float64))[0])):
    centered = M171 - M171.mean(axis=0)
    pca = PCA(n_components=10).fit(centered)
    scores = pca.transform(centered)
    rs = [abs(float(pearsonr(scores[matched, k], valence).statistic)) for k in range(10)]
    best_pc = int(np.argmax(rs)) + 1
    print(f"{name:12s}: valence best at PC{best_pc} (|r|={max(rs):.3f}); "
          f"per-PC |r| {[round(r, 2) for r in rs[:5]]}")

unprojected : valence best at PC3 (|r|=0.762); per-PC |r| [0.09, 0.1, 0.76, 0.29, 0.07]
projected   : valence best at PC2 (|r|=0.734); per-PC |r| [0.35, 0.73, 0.03, 0.16, 0.04]


## Verdicts

Written after the cells above ran; the recorded verdicts live in TREE.md (Q1.H2.E6, E7) and
follow the printed numbers, not impressions. See the research log for the day's narrative.